# Pose Estimation Pipeline: Teacher -> Student

Этот notebook запускает полный Kaggle-пайплайн проекта:

1. подключение кода репозитория;
2. проверка filtered pedestrian crops;
3. генерация pseudo-labels Teacher-моделью ViTPose;
4. фильтрация pseudo-labels;
5. сборка COCO-like датасета;
6. обучение компактной Student-модели;
7. визуализация Student predictions и speed benchmark.

Данные и веса сохраняются в `/kaggle/working`, поэтому после важных этапов нужно архивировать результаты.

## 1. Загрузка кода

Код проекта берется из GitHub. Данные не лежат в репозитории, они подключаются как Kaggle Dataset.

In [ ]:
!rm -rf /kaggle/working/pose-estimation
!git clone https://github.com/abbos-trnv/pose-estimation.git /kaggle/working/pose-estimation
%cd /kaggle/working/pose-estimation

## 2. Пути

`FILTERED_CROPS_DIR` должен указывать на Kaggle Dataset с папкой `nuscenes_pedestrian_crops_filtered`.

In [ ]:
from pathlib import Path

FILTERED_CROPS_DIR = Path(
    "/kaggle/input/datasets/abbosturgunov/pose-dataset/nuscenes_pedestrian_crops_filtered"
)

TEACHER_OUTPUT_DIR = Path("/kaggle/working/data/pseudo/nuscenes_pose_teacher")
KEYPOINT_QA_DIR = Path("/kaggle/working/data/qa/keypoints_preview")
COCO_OUTPUT_DIR = Path("/kaggle/working/data/pseudo/nuscenes_pose_coco")
STUDENT_OUTPUT_DIR = Path("/kaggle/working/models/student_baseline")
STUDENT_QA_DIR = Path("/kaggle/working/data/qa/student_predictions")
DEVICE = "cuda:0"

## 3. Проверка входных данных

Проверяем, что manifest и кропы доступны.

In [ ]:
!ls -lah {FILTERED_CROPS_DIR}
!head -1 {FILTERED_CROPS_DIR / "manifest.jsonl"}
!find {FILTERED_CROPS_DIR / "crops"} -type f | head

## 4. Установка зависимостей Teacher

Используем HuggingFace ViTPose, чтобы не зависеть от MMPose/MMCV в Kaggle Python 3.12.

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate safetensors pillow

## 5. Teacher smoke inference

Сначала запускаем Teacher только на 128 кропах. Это обязательная проверка перед full inference.

In [ ]:
!python scripts/run_hf_vitpose_teacher.py   --dataset-dir {FILTERED_CROPS_DIR}   --output-dir {TEACHER_OUTPUT_DIR}_smoke   --model-name usyd-community/vitpose-plus-base   --device {DEVICE}   --limit 128

## 6. QA smoke keypoints

Зеленый/желтый skeleton должен лежать на теле человека. Если Teacher явно ошибается на большинстве примеров, full inference запускать нельзя.

In [ ]:
!python scripts/visualize_keypoints.py   --dataset-dir {FILTERED_CROPS_DIR}   --labels-path {TEACHER_OUTPUT_DIR}_smoke/pseudo_labels.jsonl   --output-dir {KEYPOINT_QA_DIR}_smoke   --num-samples 64

from IPython.display import Image, display

display(Image(filename=str(KEYPOINT_QA_DIR) + "_smoke/keypoint_grid.jpg"))

## 7. Фильтрация smoke pseudo-labels

Baseline-фильтр оставляет примеры с `mean_keypoint_score >= 0.5` и минимум 8 уверенными keypoints.

In [ ]:
!python scripts/filter_pseudo_labels.py   --labels-path {TEACHER_OUTPUT_DIR}_smoke/pseudo_labels.jsonl   --output-dir {TEACHER_OUTPUT_DIR}_smoke_filtered   --config configs/filtering/pseudo_labels_default.json

!cat {TEACHER_OUTPUT_DIR}_smoke_filtered/report.json

## 8. Full Teacher inference

Запускается после успешного smoke QA. На текущем датасете около 37k кропов.

In [ ]:
!python scripts/run_hf_vitpose_teacher.py   --dataset-dir {FILTERED_CROPS_DIR}   --output-dir {TEACHER_OUTPUT_DIR}   --model-name usyd-community/vitpose-plus-base   --device {DEVICE}

## 9. Фильтрация full pseudo-labels

После full inference фильтруем явный шум и сохраняем отчет.

In [ ]:
!python scripts/filter_pseudo_labels.py   --labels-path {TEACHER_OUTPUT_DIR}/pseudo_labels.jsonl   --output-dir {TEACHER_OUTPUT_DIR}_filtered   --config configs/filtering/pseudo_labels_default.json

!cat {TEACHER_OUTPUT_DIR}_filtered/report.json

## 10. QA full filtered keypoints

Проверяем случайную выборку уже после фильтрации.

In [ ]:
!python scripts/visualize_keypoints.py   --dataset-dir {FILTERED_CROPS_DIR}   --labels-path {TEACHER_OUTPUT_DIR}_filtered/pseudo_labels_filtered.jsonl   --output-dir {KEYPOINT_QA_DIR}_filtered   --num-samples 64

display(Image(filename=str(KEYPOINT_QA_DIR) + "_filtered/keypoint_grid.jpg"))

## 11. Сборка COCO-like датасета

Этот датасет будет использоваться для обучения Student.

In [ ]:
!python scripts/build_coco_pose_dataset.py   --dataset-dir {FILTERED_CROPS_DIR}   --labels-path {TEACHER_OUTPUT_DIR}_filtered/pseudo_labels_filtered.jsonl   --output-dir {COCO_OUTPUT_DIR}   --copy-images

!ls -lah {COCO_OUTPUT_DIR}
!cat {COCO_OUTPUT_DIR / "report.json"}

## 12. Сохранение COCO-like датасета

Скачайте архив или загрузите его как Kaggle Dataset для повторного обучения без повторного Teacher inference.

In [ ]:
!zip -q -r /kaggle/working/nuscenes_pose_coco.zip {COCO_OUTPUT_DIR}

from IPython.display import FileLink
FileLink("/kaggle/working/nuscenes_pose_coco.zip")

## 13. Восстановление COCO датасета из архива

Если вы начали новую Kaggle-сессию и уже сохранили `nuscenes_pose_coco.zip`, восстановите его. Если `COCO_OUTPUT_DIR` уже существует, этот шаг можно пропустить.

In [ ]:
# Optional restore step.
# !mkdir -p /kaggle/working/restored
# !unzip -q /kaggle/working/nuscenes_pose_coco.zip -d /kaggle/working/restored
# COCO_OUTPUT_DIR = Path("/kaggle/working/restored/kaggle/working/data/pseudo/nuscenes_pose_coco")
# !ls -lah {COCO_OUTPUT_DIR}

## 14. Обучение Student baseline

Первый запуск на 3 эпохи нужен как smoke test. Если он проходит, запускайте 20 эпох.

In [ ]:
!python scripts/train_student.py   --config configs/training/student_baseline.json   --data-dir {COCO_OUTPUT_DIR}   --output-dir {STUDENT_OUTPUT_DIR}   --epochs 3

## 15. Полное обучение Student

Запускайте после успешного smoke training.

In [ ]:
!python scripts/train_student.py   --config configs/training/student_baseline.json   --data-dir {COCO_OUTPUT_DIR}   --output-dir {STUDENT_OUTPUT_DIR}   --epochs 20

!ls -lah {STUDENT_OUTPUT_DIR}
!cat {STUDENT_OUTPUT_DIR / "history.json"}

## 16. Визуализация Student predictions

Зеленый = Teacher pseudo-label target, красный = Student prediction.

In [ ]:
!python scripts/visualize_student_predictions.py   --data-dir {COCO_OUTPUT_DIR}   --checkpoint {STUDENT_OUTPUT_DIR / "best.pt"}   --output-dir {STUDENT_QA_DIR}   --num-samples 64

display(Image(filename=str(STUDENT_QA_DIR / "student_predictions_grid.jpg")))

## 17. Speed benchmark

Измеряем latency/FPS для batch size 1 и throughput для batch size 32.

In [ ]:
!python scripts/benchmark_student_speed.py   --checkpoint {STUDENT_OUTPUT_DIR / "best.pt"}   --output-path /kaggle/working/logs/student_speed_b1.json   --batch-size 1

!python scripts/benchmark_student_speed.py   --checkpoint {STUDENT_OUTPUT_DIR / "best.pt"}   --output-path /kaggle/working/logs/student_speed_b32.json   --batch-size 32

!cat /kaggle/working/logs/student_speed_b1.json
!cat /kaggle/working/logs/student_speed_b32.json

## 18. Сохранение Student baseline outputs

Сохраняем checkpoint, history, visual QA и speed reports.

In [ ]:
!zip -q -r /kaggle/working/student_baseline_outputs.zip   {STUDENT_OUTPUT_DIR}   {STUDENT_QA_DIR}   /kaggle/working/logs

FileLink("/kaggle/working/student_baseline_outputs.zip")